<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_8_SemiParametric_Systematics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 8 — Semi-parametric systematic uncertainties

Exercise 5 constructed the nominal hybrid likelihood through

\[
r_c(x)=\frac{p_c(x\mid\alpha=0)}{p_{\rm ref}(x)},
\qquad c\in\{S,B\}.
\]

We now add one detector-response nuisance parameter, \(\alpha_{\rm scale}\). The latent signal and background densities \(p_c(z)\) are unchanged. Only the centre of the Gaussian response \(z\to x\) is varied:

\[
x\mid z,\alpha=0 \sim
\mathcal N\!\left(z\odot s,\,\operatorname{diag}(\sigma^2)\right),
\]

\[
x\mid z,\alpha=+1 \sim
\mathcal N\!\left(1.1\,z\odot s,\,\operatorname{diag}(\sigma^2)\right),
\qquad
x\mid z,\alpha=-1 \sim
\mathcal N\!\left(0.9\,z\odot s,\,\operatorname{diag}(\sigma^2)\right).
\]

For each process and direction we train one classifier for

\[
g_c^\pm(x)
=
\frac{p_c(x\mid\alpha=\pm1)}
     {p_c(x\mid\alpha=0)}.
\]

The NSBI workspace combines the nominal ratios \(r_c(x)\), the response ratios \(g_c^\pm(x)\), the selected yields, and a unit-Gaussian constraint on \(\alpha_{\rm scale}\). We finish by comparing a profile-likelihood scan in which \(\alpha_{\rm scale}\) floats with a scan in which it is fixed to zero.

This notebook deliberately does not repeat the nominal flow, nominal ratio training, or nominal validation from Exercise 5. Run Exercise 5 first with persistent storage enabled.


In [ ]:
## ============================================================================
# Google Colab setup — run me first. Safe to re-run; a no-op off Colab.
# ============================================================================
import os, sys

# --- config -----------------------------------------------------------------
REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
N_BKG, N_SIG = 100_000_000, 20_000_000
USE_DRIVE = True
REMAKE_EVENTS = False
# ----------------------------------------------------------------------------

import subprocess
from pathlib import Path

DEPENDENCIES = [
    "pytorch-lightning",
    "onnx",
    "onnxruntime",
    "onnxscript",
    "iminuit",
    "mplhep",
    "pyarrow",
]


def run(*args, env=None):
    """Run one setup command and stop immediately if it fails."""
    subprocess.run([str(arg) for arg in args], check=True, env=env)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"

    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)

    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )

    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)

    run(sys.executable, "-m", "pip", "install", "-q", *DEPENDENCIES)

    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)

    nominal_paths = [
        Path("dataframes/background.parquet"),
        Path("dataframes/signal.parquet"),
    ]
    variation_paths = [
        Path(f"dataframes/{process}_scale_{direction}.parquet")
        for process in ["background", "signal"]
        for direction in ["up", "down"]
    ]
    generator = TUTORIAL_DIR / "generate_distributions.py"
    if REMAKE_EVENTS or not all(path.exists() for path in nominal_paths):
        run(
            sys.executable, generator,
            "--n_bkg", N_BKG,
            "--n_sig", N_SIG,
            "--with-systematics",
        )
    elif not all(path.exists() for path in variation_paths):
        run(sys.executable, generator, "--systematics-only")

print("Working dir:", os.getcwd())


## Inputs and persisted models

The nominal and varied generated samples are

\[
\begin{array}{lll}
p_B(x\mid0), & p_B(x\mid+1), & p_B(x\mid-1),\\
p_S(x\mid0), & p_S(x\mid+1), & p_S(x\mid-1).
\end{array}
\]

All six files contain the same latent columns \(z_1,\ldots,z_5\). Within a process, corresponding rows also share the same random resolution residual. The only difference between nominal, up, and down is the Gaussian response mean.

Exercise 8 loads these Exercise 5 checkpoints:

- the PRESEL classifier in models_PRESEL;
- the four-member signal/reference ensemble;
- the four-member background/reference ensemble.

If any checkpoint is absent, the notebook stops with an instruction to run Exercise 5. No nominal model is silently retrained here.


In [ ]:
import gc
import os
import pprint
from pathlib import Path

import jax
jax.config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import pyarrow.parquet as pq
import torch
from IPython.display import display

import nsbi_common_utils
from nsbi_common_utils.training.utils import load_trained_model
from utils import FEATURES, density_ratio_trainer, predict_with_model
from utils_nf import (
    accumulate_preselection_histogram,
    choose_preselection_ratio_cut,
    collect_preselected_parquet,
    sample_parquet_partition,
)

FEATURES = list(FEATURES)
SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Features: {FEATURES}")


In [ ]:
BASE_PATH = Path("dataframes")
SAMPLE_PATHS = {
    ("signal", "nominal"): BASE_PATH / "signal.parquet",
    ("signal", "up"): BASE_PATH / "signal_scale_up.parquet",
    ("signal", "down"): BASE_PATH / "signal_scale_down.parquet",
    ("background", "nominal"): BASE_PATH / "background.parquet",
    ("background", "up"): BASE_PATH / "background_scale_up.parquet",
    ("background", "down"): BASE_PATH / "background_scale_down.parquet",
}

PRESEL_MODEL_DIR = Path("models_PRESEL")
NOMINAL_RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}
SYSTEMATIC_RATIO_MODEL_DIR = {
    (process, direction): Path(
        f"models_Exercise8_{process}_scale_{direction}_vs_nominal"
    )
    for process in ["signal", "background"]
    for direction in ["up", "down"]
}
SYSTEMATIC_RATIO_PLOT_DIR = {
    key: Path(str(path).replace("models_", "plots_"))
    for key, path in SYSTEMATIC_RATIO_MODEL_DIR.items()
}
SYSTEMATIC_OUTPUT_DIR = Path("saved_densities_exercise8_systematics")

for directory in [
    *SYSTEMATIC_RATIO_MODEL_DIR.values(),
    *SYSTEMATIC_RATIO_PLOT_DIR.values(),
    SYSTEMATIC_OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Use exactly the Exercise 5 row partitions.
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.92
STREAM_BATCH_SIZE = 100_000
PRESEL_CUT_HISTOGRAM_BINS = 4_000
PRESEL_LOG_RATIO_RANGE = (-20.0, 20.0)
PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO = 250.0

MAX_SYSTEMATIC_TRAIN_EVENTS = 1_000_000
MAX_EVAL_EVENTS = 250_000

RATIO_HIDDEN_LAYERS = 4
RATIO_NEURONS = 1024
RATIO_N_EPOCHS = 50
RATIO_BATCH_SIZE = 4096
RATIO_LEARNING_RATE = 1.0e-3
RATIO_HOLDOUT_FRACTION = 0.25
RATIO_VALIDATION_FRACTION = 0.20
RATIO_PATIENCE = 10
RATIO_LOAD_IF_AVAILABLE = True
RATIO_EVALUATION_BATCH_SIZE = 100_000
RATIO_FLOOR = 1.0e-12
NOMINAL_ENSEMBLE_SIZE = 4
REFERENCE_COMPONENT_EVENTS = 125_000

missing_samples = [str(path) for path in SAMPLE_PATHS.values() if not path.exists()]
if missing_samples:
    raise FileNotFoundError(
        "Missing generated samples. Re-run the setup cell. Missing:\n"
        + "\n".join(missing_samples)
    )


## Check the detector variations

Before training, inspect the parquet metadata and a bounded \(x_1\) projection. The row counts and total expected yields are unchanged by construction. The histograms should show the coherent response shift, not a change of the latent event generator.


In [ ]:
sample_summary = []
for (process, variation), path in SAMPLE_PATHS.items():
    parquet = pq.ParquetFile(path)
    sample_summary.append(
        {
            "process": process,
            "variation": variation,
            "rows": parquet.metadata.num_rows,
            "row_groups": parquet.metadata.num_row_groups,
            "path": str(path),
        }
    )
display(pd.DataFrame(sample_summary))


def first_parquet_batch(path, columns, n_rows=100_000):
    parquet = pq.ParquetFile(path)
    batch = next(parquet.iter_batches(batch_size=int(n_rows), columns=columns))
    return batch.to_pandas()


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, process in zip(axes, ["background", "signal"]):
    batches = {
        variation: first_parquet_batch(
            SAMPLE_PATHS[(process, variation)], ["x1"]
        )
        for variation in ["nominal", "up", "down"]
    }
    combined = np.concatenate(
        [batch["x1"].to_numpy() for batch in batches.values()]
    )
    bins = np.linspace(
        np.quantile(combined, 0.005), np.quantile(combined, 0.995), 61
    )
    for variation, color, linestyle in [
        ("down", "C0", "--"),
        ("nominal", "black", "-"),
        ("up", "C3", "--"),
    ]:
        ax.hist(
            batches[variation]["x1"],
            bins=bins,
            density=True,
            histtype="step",
            lw=2,
            ls=linestyle,
            color=color,
            label=variation,
        )
    ax.set_title(process)
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel("density")
    ax.legend()
fig.tight_layout()
plt.show()


## Load the Exercise 5 nominal models

The PRESEL score defines one fixed analysis region. The nominal hybrid ratios \(p_S/p_{\rm ref}\) and \(p_B/p_{\rm ref}\) will later be evaluated on the Asimov events and placed directly in the NSBI workspace.


In [ ]:
def require_checkpoint(path, explanation):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. {explanation}")
    return path


def as_inference_session(model_candidate):
    if isinstance(model_candidate, ort.InferenceSession):
        return model_candidate
    available = ort.get_available_providers()
    providers = [
        provider
        for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if provider in available
    ] or available
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    return ort.InferenceSession(
        model_candidate.SerializeToString(),
        sess_options=options,
        providers=providers,
    )


def load_ratio_pack(model_dir, ensemble_index):
    suffix = "" if ensemble_index is None else str(ensemble_index)
    model_path = require_checkpoint(
        Path(model_dir) / f"model{suffix}.onnx",
        "Run Exercise 5 through the nominal ratio-training cells first.",
    )
    scaler_path = require_checkpoint(
        Path(model_dir) / f"model_scaler{suffix}.bin",
        "Run Exercise 5 through the nominal ratio-training cells first.",
    )
    scaler, model = load_trained_model(str(model_path), str(scaler_path))
    return {"scaler": scaler, "model": as_inference_session(model)}


PRESEL_pack = load_ratio_pack(PRESEL_MODEL_DIR, 0)
NOMINAL_RATIO_MODELS = {
    process: [
        load_ratio_pack(NOMINAL_RATIO_MODEL_DIR[process], member)
        for member in range(NOMINAL_ENSEMBLE_SIZE)
    ]
    for process in ["signal", "background"]
}


def evaluate_ratio_packs(packs, dataframe, batch_size=100_000):
    chunks = []
    for start in range(0, len(dataframe), int(batch_size)):
        batch = dataframe.iloc[start : start + int(batch_size)][FEATURES]
        member_ratios = []
        for pack in packs:
            ratio = predict_with_model(
                batch.astype("float32", copy=False),
                scaler=pack["scaler"],
                model=pack["model"],
            )
            member_ratios.append(
                np.asarray(ratio, dtype=np.float64).reshape(-1)
            )
        chunks.append(np.mean(np.stack(member_ratios, axis=0), axis=0))
    values = np.concatenate(chunks) if chunks else np.empty(0, dtype=np.float64)
    if not np.isfinite(values).all():
        raise FloatingPointError("A density-ratio model returned NaN or infinity.")
    return np.maximum(values, RATIO_FLOOR)


def evaluate_PRESEL_ratio(feature_dataframe):
    return evaluate_ratio_packs(
        [PRESEL_pack],
        feature_dataframe,
        batch_size=RATIO_EVALUATION_BATCH_SIZE,
    )

print("Loaded PRESEL and both four-member nominal Exercise 5 ensembles.")


## Reconstruct the Exercise 5 selection

The PRESEL threshold is re-derived from the nominal flow-training partition using the same deterministic row split and target \(B/S\). The resulting threshold is then applied unchanged to all nominal, up, and down samples. This is essential: a systematic variation changes the acceptance of a fixed analysis region; the selection itself must not be retuned for each template.


In [ ]:
PRESEL_INPUT_STATS = {}
for process in ["signal", "background"]:
    _, stats = sample_parquet_partition(
        SAMPLE_PATHS[(process, "nominal")],
        features=FEATURES,
        partition="presel",
        max_events=1,
        batch_size=STREAM_BATCH_SIZE,
        presel_fraction=PRESEL_TRAIN_FRACTION,
        flow_train_fraction=FLOW_TRAIN_FRACTION,
        split_seed=SPLIT_SEED,
        reservoir_seed=SEED + (11 if process == "signal" else 22),
    )
    PRESEL_INPUT_STATS[process] = stats

PRESEL_INCLUSIVE_YIELD = {
    process: float(stats["inclusive_weight"])
    for process, stats in PRESEL_INPUT_STATS.items()
}
PRESEL_LOG_RATIO_EDGES = np.linspace(
    PRESEL_LOG_RATIO_RANGE[0],
    PRESEL_LOG_RATIO_RANGE[1],
    PRESEL_CUT_HISTOGRAM_BINS + 1,
)

histograms = {}
histogram_stats = {}
for process in ["signal", "background"]:
    histograms[process], histogram_stats[process] = (
        accumulate_preselection_histogram(
            SAMPLE_PATHS[(process, "nominal")],
            features=FEATURES,
            ratio_predictor=evaluate_PRESEL_ratio,
            log_ratio_edges=PRESEL_LOG_RATIO_EDGES,
            batch_size=STREAM_BATCH_SIZE,
            presel_fraction=PRESEL_TRAIN_FRACTION,
            flow_train_fraction=FLOW_TRAIN_FRACTION,
            split_seed=SPLIT_SEED,
        )
    )

PRESEL_RATIO_CUT, PRESEL_CUT_DIAGNOSTICS = choose_preselection_ratio_cut(
    histograms["signal"],
    histograms["background"],
    PRESEL_LOG_RATIO_EDGES,
    signal_inclusive_yield=PRESEL_INCLUSIVE_YIELD["signal"],
    background_inclusive_yield=PRESEL_INCLUSIVE_YIELD["background"],
    signal_partition_weight=histogram_stats["signal"]["partition_weight"],
    background_partition_weight=histogram_stats["background"]["partition_weight"],
    target_background_to_signal=PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO,
)

print(f"Fixed PRESEL ratio cut: r >= {PRESEL_RATIO_CUT:.6g}")
print(
    "Nominal histogram estimate at the cut: B/S = "
    f"{PRESEL_CUT_DIAGNOSTICS['histogram_background_to_signal']:.2f}"
)


In [ ]:
SELECTED_SAMPLES = {}
SELECTED_STATS = {}
SELECTED_YIELD = {}

for sample_index, ((process, variation), path) in enumerate(SAMPLE_PATHS.items()):
    samples, stats = collect_preselected_parquet(
        path,
        features=FEATURES,
        ratio_predictor=evaluate_PRESEL_ratio,
        ratio_cut=PRESEL_RATIO_CUT,
        max_train_events=MAX_SYSTEMATIC_TRAIN_EVENTS,
        max_eval_events=MAX_EVAL_EVENTS,
        batch_size=STREAM_BATCH_SIZE,
        presel_fraction=PRESEL_TRAIN_FRACTION,
        flow_train_fraction=FLOW_TRAIN_FRACTION,
        split_seed=SPLIT_SEED,
        reservoir_seed=SEED + 1000 + 100 * sample_index,
    )
    train_stats = stats["flow_train"]
    if train_stats["partition_weight"] <= 0.0:
        raise RuntimeError(f"Empty flow-training partition for {(process, variation)}.")

    efficiency = (
        train_stats["selected_weight"] / train_stats["partition_weight"]
    )
    selected_yield = PRESEL_INCLUSIVE_YIELD[process] * efficiency
    SELECTED_YIELD[(process, variation)] = float(selected_yield)

    for split, dataframe in samples.items():
        retained_weight = float(dataframe["weight"].sum())
        if len(dataframe) == 0 or retained_weight <= 0.0:
            raise RuntimeError(
                f"No selected {(process, variation)} events in {split}."
            )
        dataframe["weight"] *= selected_yield / retained_weight

    SELECTED_SAMPLES[(process, variation)] = samples
    SELECTED_STATS[(process, variation)] = stats

selection_rows = []
for process in ["background", "signal"]:
    nominal_yield = SELECTED_YIELD[(process, "nominal")]
    for variation in ["down", "nominal", "up"]:
        stats = SELECTED_STATS[(process, variation)]["flow_train"]
        selection_rows.append(
            {
                "process": process,
                "variation": variation,
                "selected events": stats["selected_events"],
                "efficiency": (
                    stats["selected_weight"] / stats["partition_weight"]
                ),
                "selected yield": SELECTED_YIELD[(process, variation)],
                "yield / nominal": (
                    SELECTED_YIELD[(process, variation)] / nominal_yield
                ),
                "retained train": len(
                    SELECTED_SAMPLES[(process, variation)]["flow_train"]
                ),
                "retained eval": len(
                    SELECTED_SAMPLES[(process, variation)]["eval"]
                ),
            }
        )
display(pd.DataFrame(selection_rows))


## Train the four systematic density ratios

For a process \(c\) and direction \(d\in\{+1,-1\}\), the numerator class is the selected varied sample and the denominator class is the selected nominal sample. Each class is normalized to unit total BCE weight, so the optimal classifier score satisfies

\[
s_{c,d}(x)
=
\frac{p_c(x\mid d)}
     {p_c(x\mid d)+p_c(x\mid0)},
\qquad
\frac{s_{c,d}(x)}{1-s_{c,d}(x)}
=
g_c^d(x).
\]

Training and validation events come from the flow-training row partition. A held-out subset is used for both requested diagnostics:

1. calibration of the classifier score;
2. reweighting closure \(p_c(x\mid0)\,g_c^d(x)\simeq p_c(x\mid d)\) in every feature.

The four networks are trained and validated sequentially to keep peak memory bounded.


In [ ]:
def build_systematic_training_dataframe(
    varied_df, nominal_df, max_events, seed
):
    n_events = min(int(max_events), len(varied_df), len(nominal_df))
    if n_events < int(max_events):
        print(
            f"Requested {int(max_events):,} events per class; "
            f"using {n_events:,}."
        )

    varied = varied_df.sample(
        n=n_events, random_state=seed
    )[FEATURES + ["weight"]].copy()
    nominal = nominal_df.sample(
        n=n_events, random_state=seed + 1
    )[FEATURES + ["weight"]].copy()

    varied["weights"] = varied["weight"]
    varied["weights_normed"] = varied["weight"] / varied["weight"].sum()
    varied["train_labels"] = 1.0

    nominal["weights"] = nominal["weight"]
    nominal["weights_normed"] = nominal["weight"] / nominal["weight"].sum()
    nominal["train_labels"] = 0.0

    return pd.concat([varied, nominal], ignore_index=True).sample(
        frac=1.0, random_state=seed + 2, ignore_index=True
    )


def evaluate_ratio_pack(pack, dataframe, batch_size=100_000):
    return evaluate_ratio_packs([pack], dataframe, batch_size=batch_size)


def train_and_validate_systematic_ratio(process, direction, seed):
    key = (process, direction)
    varied_train = SELECTED_SAMPLES[key]["flow_train"]
    nominal_train = SELECTED_SAMPLES[(process, "nominal")]["flow_train"]
    training_dataframe = build_systematic_training_dataframe(
        varied_train,
        nominal_train,
        MAX_SYSTEMATIC_TRAIN_EVENTS,
        seed,
    )

    trainer = density_ratio_trainer(
        dataset=training_dataframe,
        weights=training_dataframe["weights_normed"].to_numpy(),
        training_labels=training_dataframe["train_labels"].to_numpy(),
        features=FEATURES,
        features_scaling=FEATURES,
        sample_name=[
            f"{process} scale {direction}",
            f"{process} nominal",
        ],
        output_name=f"{process}_scale_{direction}",
        path_to_figures=f"{SYSTEMATIC_RATIO_PLOT_DIR[key]}/",
        path_to_models=f"{SYSTEMATIC_RATIO_MODEL_DIR[key]}/",
    )
    trainer.train(
        hidden_layers=RATIO_HIDDEN_LAYERS,
        neurons=RATIO_NEURONS,
        number_of_epochs=RATIO_N_EPOCHS,
        batch_size=RATIO_BATCH_SIZE,
        learning_rate=RATIO_LEARNING_RATE,
        scalerType="MinMax",
        ensemble_index=None,
        verbose=1,
        rnd_seed=seed,
        holdout_split=RATIO_HOLDOUT_FRACTION,
        validation_split=RATIO_VALIDATION_FRACTION,
        callback_patience=RATIO_PATIENCE,
        num_workers=0,
        load_trained_models=RATIO_LOAD_IF_AVAILABLE,
        calibration=False,
    )

    pack = {
        "scaler": trainer.scaler,
        "model": as_inference_session(trainer.model_NN),
    }

    calibration_figure = trainer.make_calib_plots(
        observable="score", nbins=40, ensemble_index="systematic"
    )
    display(calibration_figure)

    reweighted_figures = trainer.make_reweighted_plots(
        FEATURES, "linear", 50, ensemble_index="systematic"
    )
    if len(reweighted_figures) != len(FEATURES):
        raise RuntimeError("Expected one reweighting closure plot per feature.")
    for feature, figure in zip(FEATURES, reweighted_figures):
        print(f"{process} scale {direction}: reweighting closure in {feature}")
        display(figure)

    nominal_eval = SELECTED_SAMPLES[(process, "nominal")]["eval"]
    raw_nominal_ratio = evaluate_ratio_pack(
        pack, nominal_eval, batch_size=RATIO_EVALUATION_BATCH_SIZE
    )
    normalization = float(
        np.average(
            raw_nominal_ratio,
            weights=nominal_eval["weight"].to_numpy(dtype=np.float64),
        )
    )
    if not np.isfinite(normalization) or normalization <= 0.0:
        raise FloatingPointError(
            f"Invalid normalization for {process} scale {direction}."
        )
    print(
        f"{process} scale {direction}: "
        f"E_nominal[raw ratio] = {normalization:.6f}"
    )

    del (
        trainer,
        training_dataframe,
        calibration_figure,
        reweighted_figures,
        raw_nominal_ratio,
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pack, normalization


In [ ]:
SYSTEMATIC_RATIO_MODELS = {}
SYSTEMATIC_RATIO_NORMALIZATION = {}

for process_index, process in enumerate(["signal", "background"]):
    for direction_index, direction in enumerate(["up", "down"]):
        print("\n" + "=" * 80)
        print(f"Training {process}: scale {direction} / nominal")
        print("=" * 80)
        pack, normalization = train_and_validate_systematic_ratio(
            process,
            direction,
            SEED + 20_000 + 1000 * process_index + 100 * direction_index,
        )
        SYSTEMATIC_RATIO_MODELS[(process, direction)] = pack
        SYSTEMATIC_RATIO_NORMALIZATION[(process, direction)] = normalization


## Prepare the nominal Asimov dataset and all workspace arrays

The Asimov events are the independent nominal evaluation reservoirs at \(\mu_A=1\) and \(\alpha_A=0\). Their weights sum to the nominal selected signal and background yields.

The Exercise 5 nominal ratios are normalized with an equal signal/background Monte Carlo realization of \(p_{\rm ref}\). The systematic ratios are normalized on the corresponding nominal process:

\[
\mathbb E_{p_{\rm ref}}[r_c]=1,
\qquad
\mathbb E_{p_c(\cdot\mid0)}[g_c^\pm]=1.
\]

The latter separates the conditional shape ratio from the acceptance variation. The selected-yield factors \(\nu_c^\pm/\nu_c^0\) enter the workspace independently.


In [ ]:
nominal_eval = {
    process: SELECTED_SAMPLES[(process, "nominal")]["eval"].copy()
    for process in ["signal", "background"]
}
asimov_dataset = pd.concat(
    [nominal_eval["background"], nominal_eval["signal"]],
    ignore_index=True,
)
asimov_weights = asimov_dataset["weight"].to_numpy(dtype=np.float64)


def make_reference_normalization_sample(n_per_component, seed):
    pieces = []
    for index, process in enumerate(["signal", "background"]):
        source = nominal_eval[process]
        n_events = min(int(n_per_component), len(source))
        part = source.sample(
            n=n_events, random_state=seed + index
        )[FEATURES + ["weight"]].copy()
        part["reference_weight"] = 0.5 * part["weight"] / part["weight"].sum()
        pieces.append(part)
    return pd.concat(pieces, ignore_index=True)


reference_normalization_sample = make_reference_normalization_sample(
    REFERENCE_COMPONENT_EVENTS, SEED + 30_000
)
NOMINAL_RATIO_NORMALIZATION = {}
NOMINAL_RATIOS_ASIMOV = {}
for process in ["signal", "background"]:
    raw_reference_ratio = evaluate_ratio_packs(
        NOMINAL_RATIO_MODELS[process],
        reference_normalization_sample,
        batch_size=RATIO_EVALUATION_BATCH_SIZE,
    )
    normalization = float(
        np.sum(
            reference_normalization_sample["reference_weight"].to_numpy()
            * raw_reference_ratio
        )
    )
    NOMINAL_RATIO_NORMALIZATION[process] = normalization
    NOMINAL_RATIOS_ASIMOV[process] = (
        evaluate_ratio_packs(
            NOMINAL_RATIO_MODELS[process],
            asimov_dataset,
            batch_size=RATIO_EVALUATION_BATCH_SIZE,
        )
        / normalization
    )
    print(
        f"{process:10s}: E_ref[raw nominal ratio] = {normalization:.6f}"
    )

SYSTEMATIC_RATIOS_ASIMOV = {}
for process in ["signal", "background"]:
    for direction in ["up", "down"]:
        key = (process, direction)
        SYSTEMATIC_RATIOS_ASIMOV[key] = (
            evaluate_ratio_pack(
                SYSTEMATIC_RATIO_MODELS[key],
                asimov_dataset,
                batch_size=RATIO_EVALUATION_BATCH_SIZE,
            )
            / SYSTEMATIC_RATIO_NORMALIZATION[key]
        )

ARRAY_PATHS = {
    "weights": SYSTEMATIC_OUTPUT_DIR / "weights_asimov.npy",
    ("signal", "nominal"): SYSTEMATIC_OUTPUT_DIR / "ratio_signal_nominal.npy",
    ("background", "nominal"): SYSTEMATIC_OUTPUT_DIR / "ratio_background_nominal.npy",
}
for process in ["signal", "background"]:
    for direction in ["up", "down"]:
        ARRAY_PATHS[(process, direction)] = (
            SYSTEMATIC_OUTPUT_DIR
            / f"ratio_{process}_scale_{direction}_over_nominal.npy"
        )

np.save(ARRAY_PATHS["weights"], asimov_weights)
for process in ["signal", "background"]:
    np.save(ARRAY_PATHS[(process, "nominal")], NOMINAL_RATIOS_ASIMOV[process])
    for direction in ["up", "down"]:
        np.save(
            ARRAY_PATHS[(process, direction)],
            SYSTEMATIC_RATIOS_ASIMOV[(process, direction)],
        )

nominal_expected_yield = sum(
    SELECTED_YIELD[(process, "nominal")]
    for process in ["signal", "background"]
)
print(f"Asimov events: {len(asimov_dataset):,}")
print(f"Sum of Asimov weights: {asimov_weights.sum():.6f}")
print(f"Nominal expected yield: {nominal_expected_yield:.6f}")
print(
    "Weight closure: "
    f"{asimov_weights.sum() - nominal_expected_yield:+.3e}"
)


## Build the semi-parametric NSBI workspace

For process \(c\), the workspace represents the product

\[
\frac{p_c(x\mid\alpha)}{p_{\rm ref}(x)}
=
\frac{p_c(x\mid0)}{p_{\rm ref}(x)}
\times g_c(x\mid\alpha),
\]

where \(g_c\) is interpolated from its down, nominal, and up anchors using the HistFactory strategy-5 interpolation. The same nuisance parameter also interpolates the total selected yield.

The complete unbinned intensity is

\[
\frac{\nu(x\mid\mu,\alpha)}{p_{\rm ref}(x)}
=
\mu\,\nu_S(\alpha)\,r_S(x)\,g_S(x\mid\alpha)
+
\nu_B(\alpha)\,r_B(x)\,g_B(x\mid\alpha).
\]

The likelihood includes the extended-rate term, the weighted unbinned event term, and the auxiliary constraint \(\alpha_{\rm scale}^2\) in \(-2\log L\).


In [ ]:
def make_systematics_workspace():
    samples = []
    for process in ["signal", "background"]:
        nominal_yield = SELECTED_YIELD[(process, "nominal")]
        modifiers = []
        if process == "signal":
            modifiers.append(
                {"name": "mu", "type": "normfactor", "data": None}
            )
        modifiers.append(
            {
                "name": "scale",
                "type": "normplusshape",
                "data": {
                    "hi_data": [
                        SELECTED_YIELD[(process, "up")] / nominal_yield
                    ],
                    "lo_data": [
                        SELECTED_YIELD[(process, "down")] / nominal_yield
                    ],
                    "hi_ratio": str(ARRAY_PATHS[(process, "up")]),
                    "lo_ratio": str(ARRAY_PATHS[(process, "down")]),
                },
            }
        )
        samples.append(
            {
                "name": process,
                "data": [nominal_yield],
                "ratios": str(ARRAY_PATHS[(process, "nominal")]),
                "modifiers": modifiers,
            }
        )

    return {
        "channels": [
            {
                "name": "SR",
                "type": "unbinned",
                "weights": str(ARRAY_PATHS["weights"]),
                "samples": samples,
            }
        ],
        "measurements": [
            {
                "name": "meas",
                "config": {
                    "poi": "mu",
                    "parameters": [
                        {"name": "mu", "inits": [1.0], "bounds": [[0.0, 3.0]]},
                        {
                            "name": "scale",
                            "inits": [0.0],
                            "bounds": [[-5.0, 5.0]],
                        },
                    ],
                },
            }
        ],
        "version": "1.0.0",
    }


ws_systematics = make_systematics_workspace()
pprint.pprint(ws_systematics)


## Profile the scale nuisance parameter

The same workspace is used for both curves:

- fixed uncertainty: \(\alpha_{\rm scale}=0\) at every scan point;
- profiled uncertainty: \(\alpha_{\rm scale}\) is minimized conditionally at every tested \(\mu\).

Any broadening or deformation of \(t_\mu\) therefore comes only from profiling this detector-response uncertainty.


In [ ]:
model_systematics = nsbi_common_utils.models.sbi_parametric_model(
    workspace=ws_systematics,
    measurement_to_fit="meas",
)
list_parameters, initial_values = model_systematics.get_model_parameters()
print("Workspace parameters:", list_parameters)
print("Initial values:", np.asarray(initial_values))

inf_systematics = nsbi_common_utils.inference.inference(
    model_nll=model_systematics.model,
    initial_values=initial_values,
    list_parameters=list_parameters,
    num_unconstrained_params=model_systematics.num_unconstrained_param,
    model_grad=model_systematics.model_grad,
)

print("\n" + "=" * 50)
print("GLOBAL FIT WITH THE SCALE NUISANCE PROFILED")
print("=" * 50)
inf_systematics.perform_fit(freeze_params=[], fit_strategy=0)
print(
    dict(
        zip(
            list_parameters,
            np.asarray(inf_systematics.pulls_global_fit, dtype=float),
        )
    )
)


In [ ]:
SCAN_RANGE = (0.0, 3.0)
SCAN_POINTS = 50

scan_profiled, tmu_profiled = inf_systematics.perform_profile_scan(
    parameter_name="mu",
    freeze_params=[],
    bound_range=SCAN_RANGE,
    fit_strategy=0,
    size=SCAN_POINTS,
)
scan_fixed, tmu_fixed = inf_systematics.perform_profile_scan(
    parameter_name="mu",
    freeze_params=["scale"],
    bound_range=SCAN_RANGE,
    fit_strategy=0,
    size=SCAN_POINTS,
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    scan_fixed,
    tmu_fixed,
    lw=2,
    ls="--",
    label=r"scale fixed: \(\alpha_{\rm scale}=0\)",
)
ax.plot(
    scan_profiled,
    tmu_profiled,
    lw=2,
    label=r"scale profiled",
)
ax.axvline(1.0, color="black", ls=":", lw=1, alpha=0.7)
for level in [1.0, 4.0]:
    ax.axhline(level, color="0.75", ls=":", lw=1)
ax.set_xlim(*SCAN_RANGE)
ax.set_ylim(bottom=0.0)
ax.set_xlabel(r"$\mu_{\rm signal}$")
ax.set_ylabel(r"$t_\mu$")
ax.legend()
fig.tight_layout()
plt.show()


## What the workspace is doing

At \(\alpha_{\rm scale}=0\), every systematic ratio and yield multiplier is one, so the Exercise 5 nominal model is recovered. At \(\alpha_{\rm scale}=\pm1\), the response ratios reproduce the corresponding trained detector templates. Between the anchors the workspace performs smooth interpolation; outside them it extrapolates. The Gaussian constraint prevents an arbitrarily large response shift from absorbing signal-like differences at no cost.

The profiled curve cannot contain more information about \(\mu\) than the fixed-\(\alpha\) curve. Its change measures the degeneracy between the signal strength and this detector-response direction in the learned five-dimensional likelihood.


## Suggested exercises

1. Inspect the fitted \(\alpha_{\rm scale}\) as a function of \(\mu\), rather than only the profiled \(t_\mu\).
2. Apply the scale uncertainty to only signal or only background and compare the degradation.
3. Compare the learned systematic ratios with the analytic Gaussian-mixture response ratios available in this toy model.
4. Change the response anchors from \(\pm10\%\) to \(\pm5\%\) and test whether the same interpolation remains adequate.
5. Replace the common nuisance parameter by independent signal and background scale parameters and study their correlation with \(\mu\).
